# Brain Tumor Classification — EfficientNetB0
**Model:** EfficientNetB0 pretrained na ImageNet, fine-tuned na 4 klasach guzów mózgu.

In [1]:
import torch, torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}')

CUDA: True, GPU: NVIDIA GeForce RTX 4060


In [2]:
DATA_DIR = Path('../data/raw')
TRAIN_DIR = DATA_DIR / 'Training'
TEST_DIR = DATA_DIR / 'Testing'
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'TRAIN_DIR exists: {TRAIN_DIR.exists()}')
print(f'TEST_DIR exists: {TEST_DIR.exists()}')

Device: cuda
TRAIN_DIR exists: True
TEST_DIR exists: True


In [3]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_ds = datasets.ImageFolder(TEST_DIR, transform=val_transform)
val_size = int(0.15 * len(train_ds))
train_size = len(train_ds) - val_size
train_subset, val_subset = random_split(train_ds, [train_size, val_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
print(f'Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(test_ds)}')
print(f'Klasy: {train_ds.classes}')

Train: 4760, Val: 840, Test: 1600
Klasy: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [4]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print(f'Parametry modelu: {sum(p.numel() for p in model.parameters()):,}')

Parametry modelu: 4,012,672


In [5]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_correct = 0, 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * X.size(0)
        t_correct += (out.argmax(1) == y).sum().item()
    scheduler.step()

    model.eval()
    v_loss, v_correct = 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            v_loss += criterion(out, y).item() * X.size(0)
            v_correct += (out.argmax(1) == y).sum().item()

    history['train_loss'].append(t_loss / len(train_subset))
    history['val_loss'].append(v_loss / len(val_subset))
    history['train_acc'].append(t_correct / len(train_subset))
    history['val_acc'].append(v_correct / len(val_subset))
    print(f"Epoch {epoch+1}/{EPOCHS} | Train acc: {history['train_acc'][-1]:.4f} | Val acc: {history['val_acc'][-1]:.4f}")

Path('../models').mkdir(exist_ok=True)
torch.save(model.state_dict(), '../models/efficientnetb0_best.pth')
print('Model zapisany.')

Epoch 1/15 | Train acc: 0.8466 | Val acc: 0.9464


Epoch 2/15 | Train acc: 0.9454 | Val acc: 0.9667


Epoch 3/15 | Train acc: 0.9672 | Val acc: 0.9821


Epoch 4/15 | Train acc: 0.9813 | Val acc: 0.9881


Epoch 5/15 | Train acc: 0.9809 | Val acc: 0.9869


Epoch 6/15 | Train acc: 0.9878 | Val acc: 0.9952


... (5 outputs left out)

Epoch 12/15 | Train acc: 0.9964 | Val acc: 0.9893


Epoch 13/15 | Train acc: 0.9966 | Val acc: 0.9940


Epoch 14/15 | Train acc: 0.9968 | Val acc: 0.9929


Epoch 15/15 | Train acc: 0.9968 | Val acc: 0.9940
Model zapisany.


In [6]:
Path('../out').mkdir(exist_ok=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)
plt.savefig('../out/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        all_preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.savefig('../out/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

              precision    recall  f1-score   support

      glioma       1.00      0.82      0.90       400
  meningioma       0.89      0.99      0.94       400
     notumor       0.94      1.00      0.97       400
   pituitary       0.99      1.00      1.00       400

    accuracy                           0.95      1600
   macro avg       0.96      0.95      0.95      1600
weighted avg       0.96      0.95      0.95      1600

